# 09 — Snapshot Management: the Nessie way

Vanilla Iceberg has two snapshot-management primitives:
- `rollback_to_snapshot_id(sid)` — point the table at an earlier snapshot.
- `expire_snapshots(...)` — drop old snapshots from metadata to cap retention.

Two reasons neither shows up in this notebook the way you'd expect:

1. **PyIceberg 0.9.1 doesn't expose them** at the `Table.manage_snapshots()` level.
   `manage_snapshots()` here is limited to `create_tag`, `create_branch`,
   `remove_tag`, `remove_branch`. Both `expire_snapshots` and `rollback_to_snapshot_id`
   live as Spark stored procedures — see [10_spark_maintenance.ipynb](10_spark_maintenance.ipynb).

2. **Nessie collapses snapshot history into the catalog layer.** `metadata.json`
   carries only the current snapshot. The richer version history is the Nessie
   commit log, and rollback is done by reassigning the `main` branch.

This notebook demonstrates the Nessie-native rollback pattern, which actually works.

**Prerequisites:** Run 01–05.

In [1]:
import polars as pl
import requests

import os

from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

NESSIE_API = NESSIE_URI.replace("/iceberg", "/api/v2")
table = catalog.load_table(("demo", "events"))
table.refresh()
print("Snapshots in Iceberg metadata:", len(table.history()))

Connected to catalog: nessie


Snapshots in Iceberg metadata: 1


## 1. Confirm: only one snapshot in Iceberg metadata

Even after many appends/upserts/overwrites in 01–05, `inspect.snapshots()` returns 1.
Nessie discards earlier snapshot entries when synthesizing `metadata.json` for the
current commit.

In [2]:
snaps = pl.from_arrow(table.inspect.snapshots())
print(f"Rows in inspect.snapshots(): {len(snaps)}")
snaps.select(["snapshot_id", "committed_at", "operation"])

Rows in inspect.snapshots(): 1


snapshot_id,committed_at,operation
i64,datetime[ms],str
6746404416189507434,2026-05-15 11:06:37.249,"""append"""


## 2. The real history lives in Nessie's commit log

Every `append`, `overwrite`, `upsert`, `delete`, schema-evolve, partition-evolve from
01–05 is a Nessie commit on `main`. That's the version history you can roll over.

In [3]:
hist = requests.get(
    f"{NESSIE_API}/trees/main/history",
    params={"maxRecords": 50},
).json()["logEntries"]

print(f"Commits on main: {len(hist)}")
for e in hist[:8]:
    cm = e["commitMeta"]
    print(f"  {cm['hash'][:12]}  {cm.get('message','')[:60]}")

Commits on main: 16
  13e3024a6672  Update ICEBERG_TABLE demo.events
  a6cde488a42b  Update ICEBERG_TABLE demo.events
  b25ca7e87c7b  Update ICEBERG_TABLE demo.events
  37656d988880  Update ICEBERG_TABLE demo.events
  7e1e7cb73f06  Update ICEBERG_TABLE demo.events
  1436dd883324  Update ICEBERG_TABLE demo.events
  f1616aed6ef4  Update ICEBERG_TABLE demo.events
  8aed8041c469  Update ICEBERG_TABLE demo.events


## 3. Roll `main` back to a prior commit

Nessie API: `PUT /trees/{ref}@{expected_hash}` with the **target** Reference in the
body. This is a Git-style branch reset for the entire catalog state.

In [4]:
current_main = requests.get(f"{NESSIE_API}/trees/main").json()["reference"]
target_hash  = hist[min(5, len(hist) - 1)]["commitMeta"]["hash"]

rows_before = len(pl.from_arrow(table.scan().to_arrow()))
print(f"Rows before rollback: {rows_before}")
print(f"Reassigning main: {current_main['hash'][:12]} → {target_hash[:12]}")

r = requests.put(
    f"{NESSIE_API}/trees/main@{current_main['hash']}",
    json={"type": "BRANCH", "name": "main", "hash": target_hash},
)
r.raise_for_status()
print("main now at:", r.json()["reference"]["hash"][:12])

# Reload — metadata.json now comes from the rolled-back main HEAD.
table = catalog.load_table(("demo", "events"))
rows_after = len(pl.from_arrow(table.scan().to_arrow()))
print(f"Rows after rollback:  {rows_after}")

Rows before rollback: 11
Reassigning main: 13e3024a6672 → 1436dd883324


main now at:

 1436dd883324


Rows after rollback:  10


## 4. Roll back to a tag

Same call, with a tag's hash. If you created `release-v1` in notebook 08, you can
pin `main` to that release state and "undo" everything since.

In [5]:
tag = requests.get(f"{NESSIE_API}/trees/release-v1")
if tag.status_code == 200:
    tag_hash = tag.json()["reference"]["hash"]
    main_now = requests.get(f"{NESSIE_API}/trees/main").json()["reference"]["hash"]
    r = requests.put(
        f"{NESSIE_API}/trees/main@{main_now}",
        json={"type": "BRANCH", "name": "main", "hash": tag_hash},
    )
    r.raise_for_status()
    print("main now pinned to release-v1:", r.json()["reference"]["hash"][:12])
else:
    print("release-v1 tag not present — run notebook 08 first to create it.")

main now pinned to release-v1: 13e3024a6672


## 5. Orphan data files

Rolling `main` backwards leaves data files written by now-orphaned commits on S3.
Iceberg-level expire/rollback APIs (when available) can't see them — they're outside
current metadata. The right cleanup is `system.remove_orphan_files` via Spark.

---
**Next:** [10_spark_maintenance.ipynb](10_spark_maintenance.ipynb) — compaction, manifest rewrite, orphan-file cleanup.